# 00. 환경 설정 — 시작하기 전에

이 시리즈는 **LangChain → LangGraph → Deep Agents** 순서로
AI 에이전트의 핵심만 빠르게 체험하는 입문 과정입니다.

## 학습 목표

- `.env` 파일로 API 키를 안전하게 관리하는 방법을 익힌다
- `ChatOpenAI`로 LLM 모델을 초기화한다
- 모델에 간단한 질문을 보내 정상 동작을 확인한다

## 0.1 API 키 설정

프로젝트 루트의 `.env.example`을 `.env`로 복사하고, 아래 키를 입력하세요:

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...   # 선택
```

| 키 | 용도 | 발급처 |
|---|---|---|
| `OPENAI_API_KEY` | LLM 호출 (필수) | https://platform.openai.com/api-keys |
| `TAVILY_API_KEY` | 웹 검색 도구 (선택) | https://tavily.com |

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("\u2713 API 키 로드 완료")

✓ API 키 로드 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 0.2 모델 초기화

`ChatOpenAI`는 OpenAI 호환 LLM을 래핑하는 LangChain 클래스입니다.
이후 노트북에서 이 `model` 객체를 반복적으로 사용합니다.

In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4")
print("\u2713 모델 설정 완료:", model.model_name)

✓ 모델 설정 완료: gpt-5.4


## 0.3 동작 확인

모델에 간단한 메시지를 보내 정상적으로 응답하는지 확인합니다.

In [6]:
print(model)

profile={'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x7d1dfd519010> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7d1dfe084b00> root_client=<openai.OpenAI object at 0x7d1e1c3fa9f0> root_async_client=<openai.AsyncOpenAI object at 0x7d1dfe04c710> model_name='gpt-5.4' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [8]:
response = model.invoke("안녕하세요! 한 문장으로 답해주세요.", config=lf_config)
print("\u2713 모델 응답:", response.content)

✓ 모델 응답: 안녕하세요! 무엇을 도와드릴까요?


In [9]:
response

AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 16, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DMjFCYH3vD1GN13TO1nfYgPr7bu8I', 'service_tier': 'priority', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1d16-7dcc-7422-b02c-d2034c0f46cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 14, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'priority_cache_read': 0, 'priority': 16}, 'output_token_details': {'audio': 0, 'priority_reasoning': 0, 'priority': 14}})

## 요약

| 항목 | 내용 |
|---|---|
| 환경 변수 | `load_dotenv()`로 `.env` 파일 로드 |
| 모델 | `ChatOpenAI(model="gpt-4.1")` |
| 테스트 | `model.invoke("...")` → 응답 확인 |

### 다음 단계
→ **[01_llm_basics.ipynb](./01_llm_basics.ipynb)**: LLM의 기본 — 메시지, 프롬프트, 스트리밍을 배웁니다.
